[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/upm-classes/ai_medical_imaging_2026/blob/main/notebooks/student/NB09_explain_check.ipynb)

# NB09 · Explain and check your model
**Artificial Intelligence in Medical Imaging** · Day 5 · Lab 10 (42 min)

| | |
|---|---|
| Runtime | **GPU recommended** (T4). CPU works at 64 px with fewer epochs |
| Data | PneumoniaMNIST from MedMNIST+ at 128 px (64 px on CPU), CC BY 4.0. Explanations use the **validation** set; the test set stays untouched |
| Checkpoints | Grad-CAM maps well formed · the randomised model's maps differ from the trained model's · the shift table has intervals |
| Tiers | **Core** = Grad-CAM on 3 right and 3 wrong cases + weight-randomisation sanity check · **Plus** = shift check with intervals and a planted shortcut · **Challenge** = compare Grad-CAM with occlusion on the sanity check |

**Learning goals.** (1) Compute Grad-CAM by hand with hooks. (2) Run a sanity check that tells you whether a map depends on the model. (3) Report what happens to the metric under a shift, with an interval. (4) State what an explanation cannot tell you.

**Reminder.** A map is a hypothesis about the model, not a fact about the patient. Passing a sanity check does not make a map clinically meaningful.

In [ ]:
%pip install -q "medmnist==3.0.2"
import os, random, time, copy
from pathlib import Path
import numpy as np, matplotlib.pyplot as plt, pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F, torchvision
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import roc_auc_score
from scipy.stats import spearmanr

def set_seed(seed=42):
    random.seed(seed); os.environ["PYTHONHASHSEED"] = str(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed); torch.backends.cudnn.deterministic = True; torch.backends.cudnn.benchmark = False
SEED = 42; set_seed(SEED); Path("data").mkdir(exist_ok=True); Path("outputs").mkdir(exist_ok=True)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu"); GPU = device.type == "cuda"
SIZE = 128 if GPU else 64
EP = dict(phase1=2, phase2=4) if GPU else dict(phase1=1, phase2=2)
print("torch", torch.__version__, "| torchvision", torchvision.__version__, "| device", device, "| image size", SIZE, "| epochs", EP)
if not GPU: print("!! No GPU: 64 px and few epochs. The maps will be coarser. Runtime > Change runtime type > T4 GPU.")

## 1. Data and a model to explain

In [ ]:
def load_pneumonia(size):
    try:
        from medmnist import PneumoniaMNIST
        kw = {} if size == 28 else {"size": size}
        out = {}
        for s in ("train", "val", "test"):
            d = PneumoniaMNIST(split=s, download=True, root="data", **kw)
            out[s] = (d.imgs.reshape(len(d.imgs), size, size).astype(np.uint8), d.labels.reshape(-1).astype(np.int64))
        return out, True
    except Exception as e:
        print("!! Could not download PneumoniaMNIST:", repr(e), "-> SYNTHETIC data; the maps mean nothing")
        rng = np.random.default_rng(0); out = {}
        for s, n in (("train", 1500), ("val", 300), ("test", 300)):
            y = (rng.random(n) < 0.7).astype(np.int64); x = rng.normal(100, 25, (n, size, size)); x[y == 1, size // 4: size // 2, size // 4: size // 2] += 50
            out[s] = (x.clip(0, 255).astype(np.uint8), y)
        return out, False

data, REAL = load_pneumonia(SIZE)
to_t = lambda x: torch.from_numpy(x).float().div(255).unsqueeze(1)
X = {s: to_t(v[0]) for s, v in data.items()}; Y = {s: torch.from_numpy(v[1]) for s, v in data.items()}
print({s: tuple(X[s].shape) for s in X}, "| real:", REAL)

In [ ]:
from torchvision.models import resnet18, ResNet18_Weights

class XrayResNet(nn.Module):
    """ResNet-18 for one-channel X-rays, one logit. The first max-pool is removed so that the last feature map is twice as large
    (Grad-CAM resolution = input size / 16: 8x8 at 128 px)."""
    def __init__(self, pretrained=True):
        super().__init__()
        self.net = resnet18(weights=ResNet18_Weights.IMAGENET1K_V1 if pretrained else None)
        self.net.maxpool = nn.Identity()
        self.net.fc = nn.Linear(self.net.fc.in_features, 1)
        self.register_buffer("mean", torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1))
        self.register_buffer("std", torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1))
    def forward(self, x):
        return self.net((x.repeat(1, 3, 1, 1) - self.mean) / self.std)

def make_model(pretrained=True):
    try: return XrayResNet(pretrained)
    except Exception as e: print("!! no ImageNet weights:", repr(e), "-> random init"); return XrayResNet(False)

@torch.no_grad()
def predict(model, x, bs=256):
    model.eval(); out = []
    for i in range(0, len(x), bs): out.append(torch.sigmoid(model(x[i:i + bs].to(device))).cpu())
    return torch.cat(out).numpy().ravel()

def fit(model, epochs_head, epochs_all, seed=SEED, Xtr=None, Ytr=None, bs=64):
    """Two-phase fine-tuning as in NB07: head first (BatchNorm in eval mode), then everything with a smaller learning rate."""
    Xtr = X["train"] if Xtr is None else Xtr; Ytr = Y["train"] if Ytr is None else Ytr
    set_seed(seed); model.to(device); crit = nn.BCEWithLogitsLoss()
    def run(epochs, lr, bn_eval):
        g = torch.Generator().manual_seed(seed); loader = DataLoader(TensorDataset(Xtr, Ytr), batch_size=bs, shuffle=True, generator=g)
        opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=lr)
        for ep in range(epochs):
            model.train()
            if bn_eval:
                for m in model.modules():
                    if isinstance(m, nn.BatchNorm2d): m.eval()
            for xb, yb in loader:
                xb, yb = xb.to(device), yb.float().unsqueeze(1).to(device); opt.zero_grad(); crit(model(xb), yb).backward(); opt.step()
    for p in model.parameters(): p.requires_grad = False
    for p in model.net.fc.parameters(): p.requires_grad = True
    run(epochs_head, 1e-3, True)
    for p in model.parameters(): p.requires_grad = True
    run(epochs_all, 1e-4, False)
    return model

t0 = time.time(); model = fit(make_model(True), EP["phase1"], EP["phase2"])
p_val = predict(model, X["val"]); y_val = Y["val"].numpy()
print(f"validation AUC {roc_auc_score(y_val, p_val):.3f} | {(time.time() - t0) / 60:.1f} min")

## 2. Grad-CAM by hand
Five steps from the slides: forward pass → class score → gradient of the score at the **last convolutional block** → one weight per feature map (average of its gradient) → weighted sum, **ReLU**, upsample.

Our model has one logit: its sign tells the class. `sign=+1` explains "pneumonia", `sign=-1` explains "normal".

**TODO 1.** Complete `grad_cam`: the weights are the spatial mean of the gradients (`g.mean(dim=(2, 3), keepdim=True)`), and the map is `ReLU(sum over channels of weight * activation)`.

In [ ]:
def grad_cam(model, x, layer, sign=1.0):
    """Grad-CAM maps in 0..1 with the size of x, and the logits. x: (B,1,H,W). layer: a module of `model`."""
    model.eval(); store = {}
    def fwd_hook(module, inp, out):
        store["a"] = out
        out.register_hook(lambda g: store.__setitem__("g", g))
    h = layer.register_forward_hook(fwd_hook)
    try:
        logits = model(x.to(device))[:, 0]
        model.zero_grad(); (sign * logits).sum().backward()
    finally: h.remove()
    a, g = store["a"].detach(), store["g"].detach()                   # (B, C, h, w)
    # TODO 1a: one weight per feature map = mean of the gradient over the spatial dimensions (keep dims: shape (B, C, 1, 1))
    w = None
    # TODO 1b: cam = ReLU( sum over channels of  w * a ), keep a channel dimension: shape (B, 1, h, w)
    cam = None
    cam = F.interpolate(cam, size=x.shape[-2:], mode="bilinear", align_corners=False)[:, 0]
    mx = cam.amax(dim=(1, 2), keepdim=True); cam = torch.where(mx > 0, cam / mx.clamp_min(1e-8), cam)
    return cam.cpu().numpy(), logits.detach().cpu().numpy()

### Checkpoint 1 · well-formed maps, and the opposite class gives a different map

In [ ]:
LAYER = model.net.layer4[-1]                      # last convolutional block
xb = X["val"][:8]
cam_p, lg = grad_cam(model, xb, LAYER, +1.0); cam_n, _ = grad_cam(model, xb, LAYER, -1.0)
print("map shape:", cam_p.shape, "| range:", float(cam_p.min()), "to", float(cam_p.max()))
assert cam_p.shape == (8, SIZE, SIZE), "one map per image, same size as the image"
assert np.isfinite(cam_p).all() and cam_p.min() >= 0 and cam_p.max() <= 1 + 1e-6, "maps must be in 0..1 (ReLU, then normalise)"
assert np.abs(cam_p - cam_n).mean() > 1e-4, "the map of the opposite class should look different"
print("Checkpoint 1 passed")

### Three right and three wrong cases (validation)
We take the **most confident correct** cases and the **most confident wrong** cases, because they teach the most. Explain the class the model **predicted**.

In [ ]:
pred = (p_val >= 0.5).astype(int); conf = np.abs(p_val - 0.5)
right = np.where(pred == y_val)[0]; wrong = np.where(pred != y_val)[0]
pick = list(right[np.argsort(-conf[right])][:3]) + list(wrong[np.argsort(-conf[wrong])][:3])
print("validation errors at threshold 0.5:", len(wrong), "of", len(y_val))
assert len(wrong) >= 3, "need at least 3 wrong cases to look at"

def show_maps(model, idx, layer, titles=None, ncols=6):
    xs = X["val"][idx]; sg = torch.where(torch.tensor(p_val[idx]) >= 0.5, 1.0, -1.0)
    cams = np.stack([grad_cam(model, xs[i:i + 1], layer, float(sg[i]))[0][0] for i in range(len(idx))])
    fig, ax = plt.subplots(2, len(idx), figsize=(2.3 * len(idx), 5))
    for k, i in enumerate(idx):
        ax[0, k].imshow(X["val"][i, 0], cmap="gray"); ax[0, k].axis("off")
        ax[0, k].set_title(f"true {y_val[i]} | P(pneu) {p_val[i]:.2f}", fontsize=8)
        ax[1, k].imshow(X["val"][i, 0], cmap="gray"); ax[1, k].imshow(cams[k], cmap="jet", alpha=0.45); ax[1, k].axis("off")
    ax[0, 0].set_ylabel("image"); plt.tight_layout(); plt.show(); return cams

cams_trained = show_maps(model, pick, LAYER)

**Write three sentences** (in a text cell below): where is the heat on the correct cases? On the wrong cases? Is there any pattern that a radiologist would call plausible? Remember that the lungs fill most of the image, so *"it looks at the lungs"* is a weak statement.

## 3. The sanity check: randomise the weights
If a method is explaining the **model**, destroying the model must change the map. Make a copy with **random weights**, compute the same maps and compare them with the trained ones (Spearman rank correlation per image; near 1 = the map ignores the weights).

**TODO 2.** Randomise all layers: for every module that has `reset_parameters`, call it.

In [ ]:
def randomised_copy(model, seed=123):
    torch.manual_seed(seed); rnd = copy.deepcopy(model)
    # TODO 2: for every module m in rnd.modules() that has a reset_parameters method, call it
    return rnd

In [ ]:
rnd = randomised_copy(model)
cams_rand = show_maps(rnd, pick, rnd.net.layer4[-1])
corr = [spearmanr(cams_trained[k].ravel(), cams_rand[k].ravel())[0] for k in range(len(pick))]
print("Spearman correlation trained vs randomised, per image:", np.round(corr, 2), "| mean", round(float(np.nanmean(corr)), 2))

### Checkpoint 2 · the randomised model gives different maps

In [ ]:
same = sum(torch.equal(a, b) for a, b in zip(model.parameters(), rnd.parameters()))
assert same == 0, "some layers were not randomised: every parameter tensor must differ from the trained one"
assert np.nanmean(corr) < 0.9, "maps of the randomised model should differ from the trained ones; check that you hooked the right layer"
print("Checkpoint 2 passed: the map depends on the weights")

**Reflection questions.**
1. The maps changed when we randomised the weights. Does that prove the original maps are *right*? (No: it shows they depend on the model.)
2. The last feature map is 8×8 at 128 px. What does that limit? Could Grad-CAM show a 3-pixel nodule?
3. For the wrong cases: does the map suggest *why* the model was wrong, or only *where* it looked?
4. Cross-check with the library (optional): `pip install captum` and compare with `LayerGradCam(model, model.net.layer4[-1]).attribute(x, target=0)`. For a one-logit model the target is output index 0. Do the maps agree?

## Plus · a shift check, with intervals
PneumoniaMNIST has no patient metadata, so we cannot split by sex or scanner. We can still ask a related question: **what happens to the AUC when the images change in plausible ways?** (a different scanner is brighter, noisier, blurrier, lower-resolution). Report the AUC under each shift with a bootstrap 95% interval.

**TODO 3.** Complete `apply_shift`. All shifts must keep the tensor shape and the 0..1 range.

In [ ]:
def apply_shift(x, kind, seed=0):
    g = torch.Generator().manual_seed(seed)
    if kind == "clean": return x.clone()
    if kind == "brighter": return (x + 0.15).clamp(0, 1)
    # TODO 3a: "darker": subtract 0.15 and clamp to 0..1
    # TODO 3b: "noise": add 0.08 * torch.randn(x.shape, generator=g), then clamp
    # TODO 3c: "blur": F.avg_pool2d with kernel_size=5, stride=1, padding=2, count_include_pad=False
    # TODO 3d: "lowres": downsample to a quarter of the size and upsample back (F.interpolate, bilinear), then clamp
    raise NotImplementedError(kind)

In [ ]:
def bootstrap_ci(metric_fn, scores, labels, n_boot=1000, seed=0):
    rng = np.random.default_rng(seed); vals = []
    for _ in range(n_boot):
        i = rng.integers(0, len(labels), len(labels))
        if labels[i].min() == labels[i].max(): continue
        vals.append(metric_fn(labels[i], scores[i]))
    return np.percentile(vals, [2.5, 97.5])

rows = []
for kind in ["clean", "brighter", "darker", "noise", "blur", "lowres"]:
    xs = apply_shift(X["val"], kind); assert xs.shape == X["val"].shape and float(xs.min()) >= 0 and float(xs.max()) <= 1
    p = predict(model, xs); lo, hi = bootstrap_ci(roc_auc_score, p, y_val)
    rows.append(dict(shift=kind, AUC=round(roc_auc_score(y_val, p), 3), ci95=f"[{lo:.3f}, {hi:.3f}]", mean_P_pneumonia=round(float(p.mean()), 3)))
shift_table = pd.DataFrame(rows); shift_table["delta_vs_clean"] = (shift_table["AUC"] - shift_table.loc[0, "AUC"]).round(3)
shift_table.to_csv("outputs/shift_check.csv", index=False); shift_table

In [ ]:
assert len(shift_table) == 6 and shift_table["ci95"].str.startswith("[").all()
print("Checkpoint 3 passed: shift table with intervals saved to outputs/shift_check.csv")

**Interpret.** Which shift hurts most? Does the mean predicted probability move (the model becoming more or less *suspicious* of everything)? Is a change smaller than the interval worth reporting? Put the table in your model card (NB10).

## Plus · plant a shortcut and catch it
Zech et al. (2018) found models that identified the hospital. We recreate the idea on purpose: add a small **bright square in the top-left corner of every pneumonia image in the training set** (a "marker" that only the positive class has). Train a model. Then evaluate on validation images **with** the marker added to the positives (as in training), and on the same images **without** it. Finally look at the Grad-CAM.

In [ ]:
def add_marker(x, mask_pos):
    x = x.clone(); s = max(SIZE // 10, 4); x[mask_pos, :, :s, :s] = 1.0; return x

Xtr_sh = add_marker(X["train"], Y["train"] == 1)
t0 = time.time(); model_sh = fit(make_model(True), EP["phase1"], EP["phase2"], Xtr=Xtr_sh, Ytr=Y["train"])
X_val_with = add_marker(X["val"], Y["val"] == 1)
p_with = predict(model_sh, X_val_with); p_without = predict(model_sh, X["val"])
auc_with, auc_without = roc_auc_score(y_val, p_with), roc_auc_score(y_val, p_without)
print(f"shortcut model | AUC with the marker {auc_with:.3f} | AUC without it {auc_without:.3f} | {(time.time() - t0) / 60:.1f} min")
print("honest model   | AUC without any marker", round(roc_auc_score(y_val, p_val), 3))

In [ ]:
# [PLUS]
idx_pos = np.where(y_val == 1)[0][:4]
xs = X_val_with[idx_pos]; cams, _ = grad_cam(model_sh, xs, model_sh.net.layer4[-1], +1.0)
fig, ax = plt.subplots(1, 4, figsize=(10, 2.8))
for k in range(4):
    ax[k].imshow(xs[k, 0], cmap="gray"); ax[k].imshow(cams[k], cmap="jet", alpha=0.45); ax[k].axis("off")
fig.suptitle("Grad-CAM of the shortcut model on pneumonia images with the planted marker"); plt.show()
corner = float(np.mean([c[: SIZE // 5, : SIZE // 5].sum() / max(c.sum(), 1e-8) for c in cams]))
print(f"share of the heat that falls in the top-left corner (a fifth of the image side): {corner:.2f}  (a uniform map would give 0.04)")
assert auc_with > auc_without, "the shortcut model should do better when the marker is present"

**Discuss.** (1) A patient-level split would not have caught this. What would have? (Masking the cue; testing at another site; looking at the maps.) (2) Was the corner obvious on the map? (3) The honest model does worse than the shortcut model *with* the marker. Which one would you deploy? Which would a leaderboard pick?

## Challenge · Grad-CAM against occlusion on the same sanity check
Occlusion hides a patch, recomputes the score and records the drop. It is slow but needs no gradients. Run **both methods on the trained and on the randomised model** for the same six images and compare the Spearman correlations. Which method depends more on the weights? What does that tell you, and what does it not?

In [ ]:
# [CHALLENGE]
# TODO (Challenge): write occlusion_map(score_fn, img, patch, stride): slide a patch over the image, replace it by the mean grey value,
# record how much the score drops, and average the drops over the patches that cover each pixel.
# Then run Grad-CAM and occlusion on the trained and on the randomised model for the same six images
# and compare the mean Spearman correlations (Grad-CAM vs occlusion). Write 5 lines of interpretation.
...

## Summary
- Grad-CAM = gradient-weighted feature maps at the last convolutional block; coarse by construction.
- A sanity check (randomise the weights) tests whether a map depends on the model. It does not test whether the map is right.
- A planted shortcut shows why a good number and a plausible-looking model are not enough: mask the cue, change the site, look at the map.
- Report a shift check with an interval, and put it in the model card.

*Next:* NB10 (your project notebook) uses these functions.